# biexciton model with cavity coupling (mediated by dark states)

### Accessible states
Custom Hilbert space for a cavity coupling with excitons: {|g>,|B>,|C>,|D>,|BB>,|BC>,|BD>,|CC>,|CD>,|DD>} with g: ground, B: bright state exciton, C: cavity photon, D: dark state exciton.

In [1]:

# -*- coding: utf-8 -*-
"""
Created on Thu June 18 11:xx:xx 2026

@author: felix
"""

import sys
import time
import pickle

import itertools
import matplotlib.pyplot as plt
from qutip import *
import numpy as np
from qudpy.Classes import *
import qudpy.plot_functions as pf
import ufss
import matplotlib.animation as animation
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import axes3d
from tqdm.notebook import tqdm

np.set_printoptions(threshold=sys.maxsize, linewidth=190, precision=5, suppress=True)

plt.style.use('dark_background')

hbar = 0.658211951  # in eV fs
kB = 8.617333262 * 1e-5  # Boltzmann constant eV/K
T = 300  # temperature in K
kT = T * kB
beta = 1 / kT

### parameters

In [2]:
scale = 45
scale_en = 45

en_b=1.4907 # bright state excitonic resonant energy in eV
en_d=1.4907 # dark state excitonic resonant energy in eV
detuning = -0.004 * scale_en # detuning between exciton and cavity in eV
en_cav=en_b + detuning # cavity resonant energy in eV

order = 3

rabi = 0.003 * scale  # vaccum rabi splitting
g = rabi / 2  # coupling strength for light-matter interaction in eV
J = 0.00075 / 2 * scale  # coupling strength between heavy and light excitons (from small band gap energy difference)
n_th = 0. # average number of cavity photons

muc = 1.0  # dipole strength for cavity photons
mub = 1.0  # dipole strength for heavy excitons
mud = 0.0  # dipole strength for light excitons

mu_qm1 = 1#0.3  # dipole strength for C
mu_qm2 = 1#0.29  # dipole strength for CC
mu_d1 = 1#0.04  # dipole strength for D
mu_d2 = 1#0.25  # dipole strength for DD

kappa = 0.0011 * scale  # cavity decay strength
gamma_b_decay = 0.0005 * scale  # heavy exciton decay strength
gamma_d_decay = 0.0001 * scale  # light exciton decay strength
gamma_cav_phase = 0.0006 * scale  # cavity dephasing strength
gamma_exc_phase = 0.0001 * scale  # exciton dephasing strength
gamma_bd = 0.001 # b-d swapping strength
gamma_bc = 0.001 # b-c swapping strength

bb_bind=0.00086 * scale_en # strength of BB biexciton binding
bd_bind=0.00086 * scale_en # strength of BD biexciton binding
dd_bind=0.00086 * scale_en # strength of DD biexciton binding

model="rw" # "no_rw" for no rotating wave approximation, "rw" for rotating wave approximation
# model="no-rw"

date=time.strftime('%Y-%m-%d')
directory="QuDPy for SC/Felix Brunelle/2026/biexciton features/" # result directory

hour = time.strftime("%Hh%Mm%S")
graph_title = hour

# 0-quantum + 1-quantum + 2-quantum


In [3]:
os.makedirs(f"{directory}{date}/{graph_title}_swap_bc_bd", exist_ok=True)

rang = [0.0001,0.1,5]
for enu, (bc, bd) in tqdm(enumerate(itertools.product(np.geomspace(rang[0], rang[1], rang[2]), np.geomspace(rang[0], rang[1], rang[2])))):
    print(enu, bc, bd)
    gamma_bc = bc
    gamma_bd = bd

    # setting up dm
    exc_basis0 = np.zeros((10, 10));
    exc_basis0[0, 0] = 1;
    exc_basis = Qobj(exc_basis0)  # ground,B,C,D,BB,BC,BD,CC,CD,DD
    rho = exc_basis * exc_basis.dag()  # ground state of Hamiltonian

    wc = en_cav / hbar
    wb = en_b / hbar  # bright exciton resonant frequencies
    wd = en_d / hbar  # dark exciton resonant frequencies

    # each individual lowering operators
    b_low = np.zeros_like(exc_basis0);
    b_low[0, 1] = 1;
    b_low = Qobj(b_low)

    c_low = np.zeros_like(exc_basis0);
    c_low[0, 2] = 1;
    c_low = Qobj(c_low)

    d_low = np.zeros_like(exc_basis0);
    d_low[0, 3] = 1;
    d_low = Qobj(d_low)

    bb_low = np.zeros_like(exc_basis0);
    bb_low[1, 4] = 1;
    bb_low = Qobj(bb_low)

    bc_low = np.zeros_like(exc_basis0);
    bc_low[1, 5] = 1 / np.sqrt(2);
    bc_low = Qobj(bc_low)

    cb_low = np.zeros_like(exc_basis0);
    cb_low[2, 5] = 1 / np.sqrt(2);
    cb_low = Qobj(cb_low)

    bd_low = np.zeros_like(exc_basis0);
    bd_low[1, 6] = 1 / np.sqrt(2);
    bd_low = Qobj(bd_low)

    db_low = np.zeros_like(exc_basis0);
    db_low[3, 6] = 1 / np.sqrt(2);
    db_low = Qobj(db_low)

    cc_low = np.zeros_like(exc_basis0);
    cc_low[2, 7] = 1;
    cc_low = Qobj(cc_low)

    cd_low = np.zeros_like(exc_basis0);
    cd_low[2, 8] = 1 / np.sqrt(2);
    cd_low = Qobj(cd_low)

    dc_low = np.zeros_like(exc_basis0);
    dc_low[3, 8] = 1 / np.sqrt(2);
    dc_low = Qobj(dc_low)

    dd_low = np.zeros_like(exc_basis0);
    dd_low[3, 9] = 1;
    dd_low = Qobj(dd_low)

    # global lowering operators
    cav_low = c_low + bc_low + cc_low + dc_low
    exc_b_low = b_low + bb_low + cb_low + db_low
    exc_d_low = d_low + bd_low + cd_low + dd_low

    H0_b = b_low.dag() * b_low # b-pop
    H0_c = c_low.dag() * c_low # c-pop
    H0_d = d_low.dag() * d_low # d-pop
    H0_bb = bb_low.dag() * bb_low # bb-pop
    H0_bc = bc_low.dag() * bc_low + cb_low.dag() * cb_low # bc-pop
    H0_bd = bd_low.dag() * bd_low + db_low.dag() * db_low # bd-pop
    H0_cc = cc_low.dag() * cc_low # cc-pop
    H0_cd = cd_low.dag() * cd_low + dc_low.dag() * dc_low # cd-pop
    H0_dd = dd_low.dag() * dd_low # dd-pop

    H0 = hbar * (wb * H0_b + wc * H0_c + wd * H0_d + (2 * wb - bb_bind) * H0_bb + (wb + wc) * H0_bc + (wb + wd - bd_bind) * H0_bd + 2 * wc * H0_cc + (wc + wd) * H0_cd + (2 * wd - dd_bind ) * H0_dd)

    H_coupling = b_low.dag() * d_low  # 1Q B-D coupling
    H_coupling += 2 * (bb_low.dag() @ bd_low + cb_low.dag() * cd_low + db_low.dag() * dd_low)  # 2Q B-D coupling
    H_coupling += H_coupling.dag()


    H_int = b_low.dag() * c_low  # 1Q light-matter coupling
    H_int += 2 * (bb_low.dag() * bc_low + cb_low.dag() * cc_low + db_low.dag() * dc_low) # 2Q bright light-matter coupling
    if model == "no-rw":
        H_int += 2 * cav_low * exc_b_low  # 2Q non-RWA terms (don't conserve energy)
    elif model != "rw":
        print('Wrong model name')
    H_int += H_int.dag()
    # print(H_int)

    H = H0 + hbar * g * H_int + J * H_coupling  # total hamiltonian
    # print("H0", H0)
    # print("H_int", H_int)
    # print("H_coupling", H_coupling)
    # print("H", H)

    ad = mu_qm2 * cav_low  + (mu_qm1 - mu_qm1) * c_low + mu_d2 * exc_d_low + (mu_d1 - mu_d2) * d_low  # lowering operator
    obs = ad + ad.dag()
    # print("mud", obs)
    # print("ad", ad)

    # collapse operators
    c_cav_rel = np.sqrt(kappa * (n_th + 1)) * cav_low # cavity relaxation
    c_cav_exc = np.sqrt(kappa * n_th) * cav_low.dag() # cavity excitation
    c_mat_rel = np.sqrt(gamma_b_decay * (n_th + 1)) * exc_b_low + np.sqrt(gamma_d_decay * (n_th + 1)) * exc_d_low # exciton relaxation
    c_mat_exc = np.sqrt(gamma_b_decay * n_th) * exc_b_low.dag() + np.sqrt(gamma_d_decay * n_th) * exc_d_low.dag() # exciton excitation

    c_dep = (np.sqrt(gamma_exc_phase) * (wb * H0_b + wd * H0_d) + # B and D pop
            np.sqrt(gamma_cav_phase) * wc * H0_c + # C pop
            np.sqrt(gamma_exc_phase) * (2 * wb - bb_bind) * H0_bb + # BB pop
            (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (wb + wc) / 2 * H0_bc + # BC pop
            np.sqrt(gamma_exc_phase) * (wb + wd) * H0_bd + # BD pop
            np.sqrt(gamma_cav_phase) * 2 * wc * H0_cc + # CC pop
            (np.sqrt(gamma_cav_phase) + np.sqrt(gamma_exc_phase)) * (wc + wd) / 2 * H0_cd + # CD pop
            np.sqrt(gamma_exc_phase) * (2 * wd - dd_bind ) * H0_dd) # DD pop

    # c_b_d = np.sqrt(gamma_bd) * exc_d_low.dag() * exc_b_low # bright exciton to dark exciton transition
    c_d_b = np.sqrt(gamma_bd) * exc_b_low.dag() * exc_d_low # dark exciton to bright exciton transition

    # c_b_cav = np.sqrt(gamma_bc) * cav_low.dag() * exc_b_low # bright exciton to cavity photon transition
    c_cav_b = np.sqrt(gamma_bc) * exc_b_low.dag() * cav_low # cavity photon to bright exciton transition

    c_ops = [c_cav_rel, c_cav_exc, c_dep, c_mat_rel, c_mat_exc, c_cav_b, c_d_b]#, c_b_cav, c_b_d]
    # print(c_ops)

    rho_ss = steadystate(H, c_ops)

    print("dimensionality of Hilbert-space: ", H.shape)

    # setting up system
    syst = System(H=H, rho=rho_ss, a=ad, u=obs, c_ops=c_ops, diagonalize=True)

    en, T = H.eigenstates()
    # print("eigenenergies", en)
    # print("eigenstates", T)

    print("system has been intialized")

    d0 = 1000

    dipole, t_list, spec, freq = syst.linear_spec(d0, r=1, dir=directory, title_graph=f"lindip{enu}", progress_bar_=False, plot_graph=False)

    # with open(f"{directory}{date}/{graph_title}/parameters{i}.txt", "w", encoding="utf-8") as file:
    #     file.write(hour+"\n")
    #     file.write(f"Energies  bright={en_b:.4f}, dark={en_d:.4f}, cavity={en_cav:.4f}\n")
    #     file.write(f"detuning delta={detuning:.4}\n")
    #     file.write(f"light-matter coupling g={g:.4f}\n")
    #     file.write(f"bright-dark coupling J={J:.4f}\n")
    #     file.write(f"dipole strength  bright={mub:.1f}, dark={mud:.1f}, cavity={muc:.1f}\n")
    #     file.write(f"dephasing  excitons={gamma_exc_phase:.4f}, cavity={gamma_cav_phase:.4f}\n")
    #     file.write(f"decay  bright={gamma_b_decay:.4f}, dark={gamma_d_decay:.4f}, cavity={kappa:.4f}\n")
    #     file.write(f"binding energies  bright-bright={bb_bind:.4f}, bright-dark={bd_bind:.4f}, dark-dark={dd_bind:.4f}\n")
    #     file.write(f"swapping strength  bright-dark={gamma_bd:.4f}, bright-cavity={gamma_bc:.4f}\n")
    #     file.write(f"model={model}\n")
    #
    # with open(f"{directory}{date}/{graph_title}_swap_bc_bd/dip{enu}.pkl", "wb") as file:
    #     pickle.dump(dipole, file)
    # with open(f"{directory}{date}/{graph_title}_swap_bc_bd/tlist{enu}.pkl", "wb") as file:
    #     pickle.dump(t_list, file)
    # with open(f"{directory}{date}/{graph_title}_swap_bc_bd/spec{enu}.pkl", "wb") as file:
    #     pickle.dump(spec, file)
    # with open(f"{directory}{date}/{graph_title}_swap_bc_bd/freq{enu}.pkl", "wb") as file:
    #     pickle.dump(freq, file)

    plt.figure(figsize=(12, 6))
    plt.plot(freq, np.real(spec), label="$J_{0} = 0 \\ eV$")  # real part of the spectrum
    plt.xlim(0.5, 3.)
    plt.xticks(np.arange(0.5, 3., 0.1))
    for i in H.eigenenergies():
        plt.axvline(x=i, color='red', linestyle='--')
    plt.xlabel("Energy (eV)")
    plt.ylabel("Spectrum")
    plt.legend(fontsize=14)
    plt.grid()
    plt.savefig(f"{directory}{date}/{graph_title}_swap_bc_bd/linspec{bc}_{bd}.png")
    plt.close()

0it [00:00, ?it/s]

0 0.0001 0.0001
dimensionality of Hilbert-space:  (10, 10)
diagonalizing Hamiltonian and transforming everything into eigen-basis except rho
system initialized
system has been intialized
1 0.0001 0.0005623413251903491
dimensionality of Hilbert-space:  (10, 10)
diagonalizing Hamiltonian and transforming everything into eigen-basis except rho
system initialized
system has been intialized
2 0.0001 0.0031622776601683794
dimensionality of Hilbert-space:  (10, 10)
diagonalizing Hamiltonian and transforming everything into eigen-basis except rho
system initialized
system has been intialized
3 0.0001 0.01778279410038923
dimensionality of Hilbert-space:  (10, 10)
diagonalizing Hamiltonian and transforming everything into eigen-basis except rho
system initialized
system has been intialized
4 0.0001 0.1
dimensionality of Hilbert-space:  (10, 10)
diagonalizing Hamiltonian and transforming everything into eigen-basis except rho
system initialized
system has been intialized
5 0.0005623413251903491 0